# Day E7 — 재현성과 보고: 코드 공개, 흐름도, STROBE·RECORD·TRIPOD

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/E-ehr/day-E7.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/E-ehr/day-E7.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 분석 영수증을 코드로 남기기

In [ ]:
import json
import platform
import numpy as np
import pandas as pd
import sklearn
import statsmodels

run_record = {
    "data_source": "MIMIC-IV Clinical Database Demo",
    "data_version": "2.2",
    "data_doi": "10.13026/dp1f-ex47",
    "cohort_definition_file": "cohort_definition.md",
    "random_seed": 2026,
    "python": platform.python_version(),
    "packages": {"numpy": np.__version__, "pandas": pd.__version__,
                 "scikit-learn": sklearn.__version__, "statsmodels": statsmodels.__version__},
    "cohort_flow": {"전체 ICU 입실": 5000, "18세 이상": 4793, "첫 입실만": 2717,
                    "24시간 이상": 2218, "입실 전 투약 중 제외": 1970},
}
steps = list(run_record["cohort_flow"].items())
for (a, na), (b, nb) in zip(steps, steps[1:]):
    assert nb <= na, f"'{b}' 단계에서 사람이 늘었다 -- 정의를 확인하자"
with open("run_record.json", "w", encoding="utf-8") as f:
    json.dump(run_record, f, ensure_ascii=False, indent=2)
print(json.dumps(run_record["packages"]), "| 흐름 단계 수:", len(steps))

### 예시 2 — 작은 칸 가리기

In [ ]:
import pandas as pd

tab = pd.DataFrame({"unit": ["MICU", "SICU", "CCU", "Neuro"],
                    "n": [812, 455, 301, 7], "deaths": [97, 41, 30, 3]})
THRESHOLD = 10          # 예시 값. 실제 기준은 자기 데이터 사용 계약·IRB·기관 규칙을 따른다
def suppress(t, cols, k=THRESHOLD):
    out = t.astype(object).copy()
    for c in cols:
        out.loc[t[c] < k, c] = f"<{k}"
    return out
print(suppress(tab, ["n", "deaths"]).to_string(index=False))

### 예시 3 — 원고 간이 점검

In [ ]:
report = """
## 방법
데이터: MIMIC-IV Clinical Database Demo v2.2 (doi 10.13026/dp1f-ex47).
코호트: 18세 이상, 첫 ICU 입실, index time = ICU 입실 시각.
결측: 젖산 결측 비율을 군별로 보고하고 다중 대치를 민감도 분석으로 수행.
코드: 저장소 주소와 커밋 해시를 부록에 기재.
"""
checks = {
    "데이터 출처와 버전": ["v2.2", "version", "버전"],
    "코호트 선택 기준": ["코호트", "포함", "제외"],
    "index time 정의": ["index time"],
    "노출/결과를 찾은 코드 목록": ["ICD", "itemid", "코드 목록"],
    "결측 처리": ["결측"],
    "민감도 분석": ["민감도"],
    "코드 공개": ["커밋", "저장소"],
}
for item, keys in checks.items():
    ok = any(k in report for k in keys)
    print(("✓ " if ok else "✗ 빠짐: ") + item)